# 📈 CEM4644 · MP6B — Time series
## Homework (individual): *Four other buildings, hourly electricity in 2017*

**No coding needed.** Each grey box is one step: click ▶, wait, read the result, answer the report question. Run from top to bottom.

MP6A was a table: one row per thing. This part is a **time series**: one value per hour, from the electricity meters of 4 real buildings on North American campuses, with the site's air temperature. A time series has a **rhythm** (days, weeks, seasons) that a table does not, and a model that knows the rhythm can say what comes next. You will tell the buildings apart, forecast a week three ways, find the days that do not fit, then give the same jobs to a chat model. About 75 minutes. No GPU needed.

In [ ]:
#@title ▶ Step 0 · Run me first (1–2 minutes) { display-mode: "form" }
#@markdown Click ▶ and wait for the ✅ line. Untick *load_forecaster* to skip the pretrained forecasting model (Step 2a then has two methods instead of three).
load_forecaster = True #@param {type:"boolean"}
import importlib, os, shutil, subprocess, sys
REPO, FOLDER, PKG = "CEM4644", "mp6_tabular_timeseries", "aec_tab"
FOLDERS = ["mp6_tabular_timeseries"]          # only this lab folder is downloaded, not the whole course repository

def _git(*args):
    return subprocess.run(["git", "-C", REPO, *args], capture_output=True, text=True).returncode == 0

if os.path.isdir(REPO):                      # a copy is already here: pull the newest course code over it
    if not (_git("sparse-checkout", "set", *FOLDERS)
            and _git("fetch", "-q", "--depth", "1", "origin", "master")
            and _git("reset", "-q", "--hard", "FETCH_HEAD") and _git("clean", "-qfd")):
        shutil.rmtree(REPO, ignore_errors=True)          # broken copy: start again from scratch
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", "https://github.com/Haolan-Zhang/CEM4644.git", REPO], check=True)
    subprocess.run(["git", "-C", REPO, "sparse-checkout", "set", *FOLDERS], check=True)
for _m in [m for m in list(sys.modules) if m == PKG or m.startswith(PKG + ".")]:
    del sys.modules[_m]                      # Python caches imported code: drop it, or this cell keeps the old version
importlib.invalidate_caches()
sys.path.insert(0, os.path.abspath(os.path.join(REPO, FOLDER)))
from aec_tab import lab
lab.setup(dataset="homework", part="series", load_forecaster=load_forecaster)


## Part 1 · A time series

Four other buildings, hourly electricity in 2017. Each building uses its electricity to its own rhythm: who is in it, when, and for what.

In [ ]:
#@title ▶ Step 1a · Which building is which? { display-mode: "form" }
#@markdown Four buildings, no names: a week and a year each. They are, in some order, assembly hall, office, residence hall, school.
lab.buildings()


In [ ]:
#@title ▶ Step 1b · Your answer { display-mode: "form" }
a = "assembly hall" #@param ["assembly hall", "office", "residence hall", "school"]
b = "assembly hall" #@param ["assembly hall", "office", "residence hall", "school"]
c = "assembly hall" #@param ["assembly hall", "office", "residence hall", "school"]
d = "assembly hall" #@param ["assembly hall", "office", "residence hall", "school"]
lab.buildings_answer(a, b, c, d)


In [ ]:
#@title ▶ Step 1c · The anatomy of one building's year { display-mode: "form" }
building = "Hog_office_Gustavo: office, 6,582 m²" #@param ["Hog_office_Gustavo: office, 6,582 m²", "Moose_education_Leland: school, 49,000 m²", "Robin_lodging_Janie: residence hall, 6,193 m²", "Rat_assembly_Rolland: assembly hall, 663 m²"]
lab.anatomy(building)


> ### 📝 Report question 1
> From Step 1: which buildings did you get right, and from what (the shape of the day, the weekend, the summer)? Pick one building in Step 1c and describe its week in three sentences a facilities manager would recognise.

## Part 2 · Next week

Three ways to forecast a week: copy last week; decision trees that learned from the past weeks, the calendar and the temperature; and a **pretrained forecasting model** that has seen millions of other time series and none of ours. Each is scored against what really happened.

In [ ]:
#@title ▶ Step 2a · Forecast one week { display-mode: "form" }
building = "Hog_office_Gustavo: office, 6,582 m²" #@param ["Hog_office_Gustavo: office, 6,582 m²", "Moose_education_Leland: school, 49,000 m²", "Robin_lodging_Janie: residence hall, 6,193 m²", "Rat_assembly_Rolland: assembly hall, 663 m²"]
method = "all three" #@param ["all three", "same hour last week", "decision trees (last weeks + calendar + temperature)", "Chronos-Bolt (a pretrained forecasting model, zero-shot)"]
lab.forecast(building, method)


> ### 📝 Report question 2
> From Step 2a on all four buildings: the average miss of each method. One of these buildings forecasts far worse than the others, whichever method you use: which one, why (look at Step 1c), and what extra information would a forecaster need?

## Part 3 · The odd days

Every building has a usual day for each weekday. A day that leaves the pattern is either explained (a holiday, a closure) or worth a phone call (a fault, a meter, something left running).

In [ ]:
#@title ▶ Step 3a · Days that do not fit { display-mode: "form" }
#@markdown Lower the threshold and more days are flagged; raise it and only the strangest remain.
building = "Hog_office_Gustavo: office, 6,582 m²" #@param ["Hog_office_Gustavo: office, 6,582 m²", "Moose_education_Leland: school, 49,000 m²", "Robin_lodging_Janie: residence hall, 6,193 m²", "Rat_assembly_Rolland: assembly hall, 663 m²"]
threshold = 3.5 #@param {type:"slider", min:2, max:6, step:0.5}
lab.odd_days(building, threshold)


> ### 📝 Report question 3
> From Step 3a on two buildings: the flagged days at threshold 3.5. Which have an obvious cause (the calendar column), which do not? For one unexplained day, say what you would check first. What threshold would you set for an automatic alert, and why?

## Part 4 · The same jobs, by a chat model

**hokie.ai** (https://hokie.ai.vt.edu/, Virginia Tech's free access to GPT models, sign in with your VT account) gets four weeks of one building's hourly electricity and next week's temperature, and forecasts the week the notebook forecast in Step 2a; then the building's daily totals for the year, to find the odd days of Step 3a. You paste its reply back into the notebook, which scores it against what really happened, next to the notebook's own models. First the chat on its own, then the chat told to use its **data-analysis tool** (it writes and runs code on the files).

In [ ]:
#@title ▶ Step 4a · Ask the chat for next week { display-mode: "form" }
#@markdown The reply must list all 168 hours; if the chat stops early, ask it to continue and paste every part. Run the same prompt in two new chats and score both. If attaching files does not work, choose *paste the data into the prompt*.
building = "Hog_office_Gustavo: office, 6,582 m²" #@param ["Hog_office_Gustavo: office, 6,582 m²", "Moose_education_Leland: school, 49,000 m²", "Robin_lodging_Janie: residence hall, 6,193 m²", "Rat_assembly_Rolland: assembly hall, 663 m²"]
give = "attach the files" #@param ["attach the files", "paste the data into the prompt"]
lab.chat_forecast(building, give)


In [ ]:
#@title ▶ Step 4b · Ask the chat to use its analysis tool { display-mode: "form" }
#@markdown Pick the model the chat should train. If the reply shows no code or analysis panel, ask it again to *use your data-analysis tool*.
building = "Hog_office_Gustavo: office, 6,582 m²" #@param ["Hog_office_Gustavo: office, 6,582 m²", "Moose_education_Leland: school, 49,000 m²", "Robin_lodging_Janie: residence hall, 6,193 m²", "Rat_assembly_Rolland: assembly hall, 663 m²"]
model = "gradient-boosted trees" #@param ["gradient-boosted trees", "a random forest", "a straight line", "a small neural network"]
lab.chat_forecast_tool(building, model)


In [ ]:
#@title ▶ Step 4c · Ask the chat for the odd days { display-mode: "form" }
#@markdown The notebook compares the chat's days with the days its own rule flags in Step 3a (threshold 3.5) and with the public holidays.
building = "Hog_office_Gustavo: office, 6,582 m²" #@param ["Hog_office_Gustavo: office, 6,582 m²", "Moose_education_Leland: school, 49,000 m²", "Robin_lodging_Janie: residence hall, 6,193 m²", "Rat_assembly_Rolland: assembly hall, 663 m²"]
give = "attach the files" #@param ["attach the files", "paste the data into the prompt"]
lab.chat_odd_days(building, give)


> ### 📝 Report question 4
> Steps 4a to 4c on the building that forecast worst in Step 2a and on one other: the chat's average miss on its own and with its analysis tool against Step 2a's methods, and the odd days it found. Does the chat do better than the notebook on the hard building? Does its explanation of that building's pattern help you, and how would you check whether it is true?

## Part 5 · Your own time series

A small app, opened from a link: upload any CSV with a time column and a value column, and it forecasts the last period from the data before it.

In [ ]:
#@title ▶ Step 5 · Your own time series { display-mode: "form" }
#@markdown Open the printed link in a new tab.
lab.upload_app()


> ### 📝 Report question 5
> The main deliverable: find or make a time series of your own (a utility bill history, a site's weather, daily progress or deliveries). Run it through Step 5 and report what the data is, what the app found, and what you would need to trust the forecast. Then give the same file to the chat and ask it to use its analysis tool to forecast the same period: does it agree with the app?

## Wrap-up

In [ ]:
#@title ▶ Numbers for your report { display-mode: "form" }
lab.report_summary()


### Credits
- Meters: Building Data Genome Project 2 (hourly electricity meters and site weather, 2017), Miller et al. (2020), Scientific Data 7:368, MIT, https://github.com/buds-lab/building-data-genome-project-2.
- Pretrained forecaster: Chronos-Bolt (small) (Amazon Science, Apache-2.0).
- Chat model: the GPT models behind hokie.ai (Virginia Tech).
- Lab code: https://github.com/Haolan-Zhang/CEM4644 (folder `mp6_tabular_timeseries`).